
# 04 Transfer Learning with Torchvision: The Realistic Approach

Notebook 03 trained a CNN from scratch on 800 synthetic examples, and got suspiciously good results
because the task was artificially easy. **Real document forensics and land-use classification won't have
hundreds of thousands of labeled examples early on**, and a small CNN trained from scratch on a small
real dataset typically won't generalize well. This is the actual, standard solution used across the
industry: **start from a model already trained on millions of general images, and fine-tune only the
last part of it on your much smaller, specific dataset.**

## Why this works

A model trained on a huge, general image dataset (like ImageNet, 1.4 million images across 1,000
categories) learns, in its early layers, to detect very general visual features edges, textures,
color gradients, simple shapes. Those general features are useful for almost *any* image task, not just
the one the model was originally trained for. **Transfer learning reuses those general early layers, and
only retrains the final layers** to specialize on your specific task (genuine vs. tampered, or land-use
category).

## Loading a pretrained model

**A note before running this:** downloading pretrained weights needs internet access to
`download.pytorch.org`, which may not be reachable in every sandboxed environment. The cell below
handles that gracefully on your own machine with normal internet access, this downloads real,
useful pretrained weights the first time you run it (and caches them locally after that).


In [1]:

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet18_Weights

try:
    pretrained_model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    weights_loaded = True
    print("Loaded real ImageNet-pretrained ResNet18 weights.")
except Exception as e:
    pretrained_model = models.resnet18(weights=None)  # architecture only, random initial weights
    weights_loaded = False
    print("Could not download pretrained weights in this environment (needs internet access).")
    print("Continuing with the same architecture but random initial weights, so you can still")
    print("see the fine-tuning code pattern correctly on your own machine, this cell will")
    print("download and use real pretrained weights instead.")
    print("Error detail:", e)

print()
print(pretrained_model.fc)  # the final classification layer this is what we'll replace


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\GROUP/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:04<00:00, 11.0MB/s]


Loaded real ImageNet-pretrained ResNet18 weights.

Linear(in_features=512, out_features=1000, bias=True)



## Freezing the base, replacing the head

Two deliberate steps:

1. **Freeze every existing layer** (`requires_grad = False`) during fine-tuning, we don't want to
   destroy the useful general features these layers already learned.
2. **Replace the final fully-connected layer** with a new one matching our task (2 classes: genuine vs.
   tampered) this new layer starts with random weights and is the *only* part that gets trained.


In [2]:

for param in pretrained_model.parameters():
    param.requires_grad = False  # freeze everything first

# Replace the final layer resnet18's original final layer expects 512 input features
# (from the layers before it) and originally outputs 1000 classes (ImageNet's categories);
# we replace it with a layer outputting just our 2 classes
num_features = pretrained_model.fc.in_features
pretrained_model.fc = nn.Linear(num_features, 2)

# Only the new final layer has requires_grad=True at this point confirm that:
trainable_params = [name for name, p in pretrained_model.named_parameters() if p.requires_grad]
print("Trainable parameters (should only be the new final layer):", trainable_params)


Trainable parameters (should only be the new final layer): ['fc.weight', 'fc.bias']



## Preparing data the way a pretrained model expects it

Pretrained ImageNet models expect a specific input format: 224×224 pixels, and pixel values normalized
using ImageNet's specific mean/standard deviation per color channel. Getting this wrong (skipping
normalization, or using the wrong image size) is a common source of "why is my fine-tuned model
performing terribly" bugs.


In [3]:

import numpy as np
from torchvision import transforms

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # standard ImageNet stats
])

# Reuse the same synthetic patch generator from notebook 03
def make_patch(tampered: bool, size=64):
    base = np.random.normal(20, 5, (size, size, 3))
    if tampered:
        cy, cx = np.random.randint(16, size - 16, 2)
        yy, xx = np.ogrid[:size, :size]
        blob = np.exp(-((yy - cy)**2 + (xx - cx)**2) / (2 * 8**2)) * np.random.uniform(80, 150)
        base += blob[..., None]
    return np.clip(base, 0, 255).astype(np.uint8)

np.random.seed(5)
n_samples = 200  # deliberately small the whole point of transfer learning is needing less data
raw_patches = [make_patch(i % 2 == 0) for i in range(n_samples)]
labels = [1 if i % 2 == 0 else 0 for i in range(n_samples)]

processed = torch.stack([preprocess(p) for p in raw_patches])
labels_tensor = torch.tensor(labels, dtype=torch.long)

print("Processed batch shape:", processed.shape, " (matches the 224x224x3 input a pretrained model expects)")


Processed batch shape: torch.Size([200, 3, 224, 224])  (matches the 224x224x3 input a pretrained model expects)



## Fine-tuning

Same training loop shape as notebook 03 the only difference is which parameters the optimizer is
allowed to update.


In [5]:

from torch.utils.data import TensorDataset, DataLoader, random_split

dataset = TensorDataset(processed, labels_tensor)
train_size = int(0.8 * len(dataset))
train_ds, test_ds = random_split(dataset, [train_size, len(dataset) - train_size],
                                  generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)

# Only pass the trainable (unfrozen) parameters to the optimizer
optimizer = torch.optim.Adam(
    [p for p in pretrained_model.parameters() if p.requires_grad],
    lr=0.001,
)
loss_fn = nn.CrossEntropyLoss()

n_epochs = 10
for epoch in range(n_epochs):
    pretrained_model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = pretrained_model(batch_x)
        loss = loss_fn(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{n_epochs} loss: {total_loss/len(train_loader):.4f}")


Epoch 1/10 loss: 0.2557
Epoch 2/10 loss: 0.1596
Epoch 3/10 loss: 0.1549
Epoch 4/10 loss: 0.1308
Epoch 5/10 loss: 0.1314
Epoch 6/10 loss: 0.0773
Epoch 7/10 loss: 0.0692
Epoch 8/10 loss: 0.0704
Epoch 9/10 loss: 0.0634
Epoch 10/10 loss: 0.1050


In [6]:

pretrained_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        preds = pretrained_model(batch_x).argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

print(f"Test accuracy: {correct/total*100:.1f}% (on only {train_size} training examples)")
if not weights_loaded:
    print()
    print("Note: since real pretrained weights couldn't be downloaded in this environment,")
    print("this result doesn't demonstrate transfer learning's real advantage it's using")
    print("random initial weights, not genuinely useful pretrained features. On your own")
    print("machine with internet access, re-run this notebook to see the real benefit: good")
    print("accuracy from far fewer training examples than notebook 03 needed.")


Test accuracy: 100.0% (on only 160 training examples)



## Exercises

### Exercise 1
Try **unfreezing** the last block of the pretrained model (`pretrained_model.layer4`) in addition to the
final classification layer, and re-train. This is a common middle-ground technique ("partial
fine-tuning") more capacity to adapt than freezing everything, less risk of overfitting than
retraining the whole network from scratch on a small dataset.


In [6]:
# Your code here


#### Solution

In [7]:

for param in pretrained_model.layer4.parameters():
    param.requires_grad = True

optimizer2 = torch.optim.Adam(
    [p for p in pretrained_model.parameters() if p.requires_grad],
    lr=0.0005,  # a smaller learning rate is standard practice when unfreezing more layers
)

trainable_now = sum(p.numel() for p in pretrained_model.parameters() if p.requires_grad)
print(f"Trainable parameters after unfreezing layer4: {trainable_now:,}")


Trainable parameters after unfreezing layer4: 8,394,754



### Exercise 2
Write a short markdown explanation (no code needed) of when you'd choose **feature extraction** (freeze
everything except the final layer, as in the main notebook) versus **full fine-tuning** (unfreeze
everything). Consider: how much labeled data you have, and how similar your task is to the original
ImageNet task. (Hint: document images and satellite imagery are visually quite different from ImageNet's
everyday photos does that change your answer?)


In [8]:
# Write your answer as a comment, or switch this cell to Markdown



## What's next

You now have both approaches to document classification: rule-based forensic checks (notebook 02, usable
immediately) and a trained-model path (notebooks 03–04, for once real labeled data accumulates).

The other CV-tagged feature in the build matrix is satellite imagery, which needs one more concept these
notebooks haven't covered yet: **georeferenced raster data** images where each pixel corresponds to a
real-world location. `05_satellite_imagery_rasterio.ipynb` covers that, and reconnects back to the
GeoPandas/Shapely curriculum from before.
